# Parallel Traveling Salesperson Problem (TSP) in C++

Run the cells in order on a **CPU runtime**. All project algorithms, validation,
benchmarking and comparisons are C++17. The notebook only writes files and runs
shell commands; it installs no Python packages.

1. **Focused Ant Colony Optimization (FACO): baseline versus A+B combined.**
2. **Iterated Local Search (ILS): normal ILS versus ILS + focused repair + selective sharing.**

Both methods in each comparison use the same threads and seeds. FACO uses matched
iteration counts; ILS uses matched time budgets. These are separate experiments.
No results are prefilled: run the cells to obtain measurements on your runtime.

An **instance** is one city dataset and its distance rule. The default `pr2392`
contains 2,392 cities; bundled alternatives are `d15112` (15,112 cities) and
`Tnm30001` (30,001 cities). `EUC_2D` means rounded Euclidean distances in two
dimensions. Larger instances need more memory and longer budgets.

Research source: R. Skinderowicz, *Improving Ant Colony Optimization efficiency for
solving large TSP instances*, Applied Soft Computing (2022),
[DOI](https://doi.org/10.1016/j.asoc.2022.108653),
[FocusedACO source](https://github.com/RSkinderowicz/FocusedACO).
General ILS framework: Lourenço, Martin and Stützle,
[Iterated Local Search](https://arxiv.org/abs/math/0102188).
Our repair/sharing modification is experimental; improvements are not guaranteed.


## 1. Create the project directory
Existing results are retained.


In [ ]:
%%bash
set -euo pipefail
mkdir -p /content/parallel_tsp_cpp


## 2. C++ experiment settings
Edit these values before compiling. `thread_count` controls Open Multi-Processing
(OpenMP) workers. The two ILS methods both retain eight search trajectories.
The default ILS comparison runs 3 seeds × 2 methods × 10 seconds, plus overhead.


In [ ]:
%%writefile /content/parallel_tsp_cpp/settings.hpp
#pragma once

#include <bits/stdc++.h>
#include <omp.h>

using namespace std;

struct ExperimentSettings {
    string instance_name = "pr2392";
    int thread_count = min(4, omp_get_num_procs());
    vector<int> seeds = {11, 23, 37};
    int faco_iterations = 2000;
    int ant_count = 128;
    double ils_seconds = 10;
    int check_rounds = 1000;
};


## 3. FACO: baseline and A+B implementation
`run_focused_aco<false>` is baseline; `<true>` applies A+B. The compiler removes
unused template branches, so there is no per-candidate runtime mode test.

**A — direct indexing:** aligned candidate slots make pheromone access constant
time. Score preparation changes from O(n k²) to O(n k), for n cities and k candidates.
This is a stage-level improvement, not the complexity of the complete solver.

**B — merged passes:** find `One writer per row` in the code. For each candidate,
evaporate its pheromone, reinforce it if it is the selected tour's predecessor or
successor, then store its next score. A separate initialization computes iteration
zero's scores. Barriers protect tour selection and score reuse. Baseline keeps its
separate evaporation, tour reinforcement and score-preparation passes.


In [ ]:
%%writefile /content/parallel_tsp_cpp/faco.cpp
/**
 * This is an implementation of the Focused Ant Colony Optimization (FACO) for
 * solving large TSP instances as described in the paper:
 *
 * R. Skinderowicz,
 * Improving Ant Colony Optimization efficiency for solving large TSP instances,
 * Applied Soft Computing, 2022, 108653, ISSN 1568-4946,
 * https://doi.org/10.1016/j.asoc.2022.108653.
 *
 * @author: Rafał Skinderowicz (rafal.skinderowicz@us.edu.pl)
 */
#include <bits/stdc++.h>
#include <omp.h>

#include "problem_instance.h"
#include "ant.h"
#include "pheromone.h"
#include "local_search.h"
#include "utils.h"
#include "rand.h"
#include "progargs.h"
#include "json.hpp"
#include "logging.h"

using namespace std;

namespace fs = filesystem;

struct HeuristicData {
    const ProblemInstance &problem_;
    double beta_;

    HeuristicData(const ProblemInstance &instance, double beta)
        : problem_(instance), beta_(beta) {}

    [[nodiscard]] double get(uint32_t from, uint32_t to) const {
        auto d = problem_.get_distance(from, to);
        return (d > 0) ? 1. / pow(d, beta_) : 1;
    }

    [[nodiscard]] uint32_t
    find_node_with_max_value(uint32_t from, const vector<uint32_t> &nodes) const {
        assert(beta_ > 0);

        auto result = from;
        auto min_dist = numeric_limits<double>::max();
        for (auto node : nodes) {
            auto dist = problem_.get_distance(from, node);
            if (dist < min_dist) {
                min_dist = dist;
                result = node;
            }
        }
        return result;
    }
};

uint32_t select_max_product_node(uint32_t current_node, Ant &ant,
                                 const CandListPheromone & /*pheromone*/,
                                 const HeuristicData &heuristic) {

    assert(ant.get_unvisited_count() > 0);
    // We are assuming that all nodes on the cand list of the current_node
    // have been visited and thus we do not need to look for pheromone values
    // as all the other edges have the same - default - value
    return heuristic.find_node_with_max_value(current_node, ant.get_unvisited_nodes());
}

static const uint32_t MaxCandListSize = 32;

struct Limits {
    double min_ = 0;
    double max_ = 0;
};

/**
 * This is based on Eq. 11 from the original MAX-MIN paper:
 *
 * Stützle, Thomas, and Holger H. Hoos. "MAX–MIN ant system." Future generation
 * computer systems 16.8 (2000): 889-914.
 */
Limits calc_trail_limits(uint32_t dimension, uint32_t /*cand_list_size*/, double p_best,
                         double rho, double solution_cost) {
    const auto tau_max = 1 / (solution_cost * (1. - rho));
    const auto cand_count = dimension;
    const auto avg = cand_count / 2.;
    const auto p = pow(p_best, 1. / cand_count);
    const auto tau_min = min(tau_max, tau_max * (1 - p) / ((avg - 1) * p));
    return {tau_min, tau_max};
}

/**
 * This is a modified version of the original trail initialization method
 * used in the FACO
 */
Limits calc_trail_limits_cl(uint32_t /*dimension*/, uint32_t cand_list_size,
                            double p_best, double rho, double solution_cost) {
    const auto tau_max = 1 / (solution_cost * (1. - rho));
    const auto avg = cand_list_size; // This is far smaller than dimension/2
    const auto p = pow(p_best, 1. / avg);
    const auto tau_min = min(tau_max, tau_max * (1 - p) / ((avg - 1) * p));
    return {tau_min, tau_max};
}

typedef Limits (*calc_trail_limits_fn_t)(uint32_t dimension,
                                         uint32_t /*cand_list_size*/, double p_best,
                                         double rho, double solution_cost);

template <typename Pheromone_t>
uint32_t select_next_node(const Pheromone_t &pheromone, const HeuristicData &heuristic,
                          const NodeList &nn_list,
                          const vector<double> &nn_product_cache,
                          const NodeList &backup_nn_list, Ant &ant) {
    assert(!ant.route_.empty());

    const auto current_node = ant.get_current_node();
    assert(nn_list.size() <= ::MaxCandListSize);

    // A list of the nearest unvisited neighbors of current_node, i.e. so
    // called "candidates list", or "cl" in short
    uint32_t cl[::MaxCandListSize];
    uint32_t cl_size = 0;

    // In the MMAS the local pheromone evaporation is absent thus for each ant
    // the product of the pheromone trail and the heuristic will be the same
    // and we can pre-load it into nn_product_cache
    auto nn_product_cache_it =
        nn_product_cache.begin() + static_cast<uint32_t>(current_node * nn_list.size());

    double cl_product_prefix_sums[::MaxCandListSize];
    double cl_products_sum = 0;
    double max_prod = 0;
    uint32_t max_node = current_node;
    for (auto node : nn_list) {
        uint32_t valid = 1 - ant.is_visited(node);
        cl[cl_size] = node;
        auto prod = *nn_product_cache_it * valid;
        cl_products_sum += prod;
        cl_product_prefix_sums[cl_size] = cl_products_sum;
        cl_size += valid;
        ++nn_product_cache_it;
        if (max_prod < prod) {
            max_prod = prod;
            max_node = node;
        }
    }

    uint32_t chosen_node = max_node;

    if (cl_size > 1) { // Select from the closest nodes
        // The following could be done using binary search in O(log(cl_size))
        // time but should not matter for small values of cl_size
        chosen_node = cl[cl_size - 1];
        const auto r = get_rng().next_float() * cl_products_sum;
        for (uint32_t i = 0; i < cl_size; ++i) {
            if (r < cl_product_prefix_sums[i]) {
                chosen_node = cl[i];
                break;
            }
        }
    } else if (cl_size == 0) { // Select from the rest of the unvisited nodes the one
                               // with the maximum product of pheromone and heuristic
        for (auto node : backup_nn_list) {
            if (!ant.is_visited(node)) {
                chosen_node = node;
                break;
            }
        }
        if (chosen_node == max_node) { // Still nothing selected
            chosen_node =
                select_max_product_node(current_node, ant, pheromone, heuristic);
        }
    }
    assert(chosen_node != current_node);
    return chosen_node;
}

/**
 * This wraps problem instance and pheromone memory and provides convenient
 * methods to manipulate it.
 */
template <class Impl> class ACOModel {
  protected:
    const ProblemInstance &problem_;
    double p_best_;
    double rho_;
    uint32_t cand_list_size_;

  public:
    Limits trail_limits_;

    calc_trail_limits_fn_t calc_trail_limits_ = calc_trail_limits;

    ACOModel(const ProblemInstance &problem, const ProgramOptions &options)
        : problem_(problem), p_best_(options.p_best_), rho_(options.rho_),
          cand_list_size_(options.cand_list_size_) {}

    void init(double solution_cost) {
        update_trail_limits(solution_cost);
        static_cast<Impl *>(this)->init_impl();
    }

    void update_trail_limits(double solution_cost) {
        trail_limits_ = calc_trail_limits_(problem_.dimension_, cand_list_size_,
                                           p_best_, rho_, solution_cost);
    }

    void evaporate_pheromone() {
        get_pheromone().evaporate(1 - rho_, trail_limits_.min_);
    }

    decltype(auto) get_pheromone() {
        return static_cast<Impl *>(this)->get_pheromone_impl();
    }

    // Increases amount of pheromone on trails corresponding edges of the
    // given solution (sol). Returns deposited amount.
    double deposit_pheromone(const Ant &sol) {
        const double deposit = 1.0 / sol.cost_;
        auto prev_node = sol.route_.back();
        auto &pheromone = get_pheromone();
        for (auto node : sol.route_) {
            // The global update of the pheromone trails
            pheromone.increase(prev_node, node, deposit, trail_limits_.max_);
            prev_node = node;
        }
        return deposit;
    }
};

class CandListModel : public ACOModel<CandListModel> {
    unique_ptr<CandListPheromone> pheromone_ = nullptr;

  public:
    CandListModel(const ProblemInstance &problem, const ProgramOptions &options)
        : ACOModel(problem, options) {}

    CandListPheromone &get_pheromone_impl() { return *pheromone_; }

    void init_impl() {
        pheromone_ =
            make_unique<CandListPheromone>(problem_.get_nn_lists(cand_list_size_),
                                           trail_limits_.max_, problem_.is_symmetric_);
    }
};

vector<vector<uint32_t>> par_build_initial_routes(const ProblemInstance &problem,
                                                  bool use_local_search,
                                                  uint32_t sol_count = 0) {
    uint32_t nn_count = 16;

    if (sol_count == 0) {
#pragma omp parallel default(none) shared(sol_count)
#pragma omp master
        sol_count = omp_get_num_procs();
    }

    vector<vector<uint32_t>> routes(sol_count);

    for (uint32_t i = 0; i < sol_count; ++i) {
        auto start_node = get_rng().next_uint32(problem.dimension_);
        routes[i] = problem.build_nn_tour(start_node);
    }

    if (use_local_search) {
#pragma omp parallel for default(none) shared(problem, routes, nn_count, sol_count)
        for (uint32_t i = 0; i < sol_count; ++i) {
            two_opt_nn(problem, routes[i], true, nn_count);
            three_opt_nn(problem, routes[i], /*use_dont_look_bits*/ true, nn_count);
        }
    }
    return routes;
}

template <bool use_combined_optimizations, typename ComputationsLog_t>
unique_ptr<Solution> run_focused_aco(const ProblemInstance &problem,
                                     const ProgramOptions &opt,
                                     ComputationsLog_t &comp_log) {

    const auto dimension = problem.dimension_;
    const auto cl_size = opt.cand_list_size_;
    const auto bl_size = opt.backup_list_size_;
    const auto ants_count = opt.ants_count_;
    const auto iterations = opt.iterations_;
    const auto use_ls = opt.local_search_ != 0;

    Timer start_sol_timer;
    const auto start_routes = par_build_initial_routes(problem, use_ls);
    auto start_sol_count = start_routes.size();
    vector<double> start_costs(start_sol_count);

#pragma omp parallel default(none)                                                     \
    shared(start_sol_count, problem, start_costs, start_routes)
#pragma omp for
    for (size_t i = 0; i < start_sol_count; ++i) {
        start_costs[i] = problem.calculate_route_length(start_routes[i]);
    }
    comp_log("initial solutions build time", start_sol_timer.get_elapsed_seconds());

    auto smallest_pos =
        distance(begin(start_costs), min_element(begin(start_costs), end(start_costs)));
    auto initial_cost = start_costs[smallest_pos];
    const auto &start_route = start_routes[smallest_pos];
    comp_log("initial sol cost", initial_cost);

    HeuristicData heuristic(problem, opt.beta_);
    vector<double> cl_heuristic_cache;

    cl_heuristic_cache.resize(cl_size * dimension);
    for (uint32_t node = 0; node < dimension; ++node) {
        auto cache_it = cl_heuristic_cache.begin() + node * cl_size;

        for (auto &nn : problem.get_nearest_neighbors(node, cl_size)) {
            *cache_it++ = heuristic.get(node, nn);
        }
    }

    // Probabilistic model based on pheromone trails:
    CandListModel model(problem, opt);
    // If the LS is on, the differences between pheromone trails should be
    // smaller -- we use calc_trail_limits_cl instead of calc_trail_limits
    model.calc_trail_limits_ = !use_ls ? calc_trail_limits : calc_trail_limits_cl;
    model.init(initial_cost);
    auto &pheromone = model.get_pheromone();
    pheromone.set_all_trails(model.trail_limits_.max_);

    vector<double> nn_product_cache(dimension * cl_size);

    auto best_ant = make_unique<Ant>(start_route, initial_cost);

    vector<Ant> ants(ants_count);
    Ant *iteration_best = nullptr;

    auto source_solution = make_unique<Solution>(start_route, best_ant->cost_);

    // The following are mainly for reporting purposes
    int64_t select_next_node_calls = 0;
    Trace<ComputationsLog_t, SolutionCost> best_cost_trace(comp_log, "best sol cost",
                                                           iterations, 1, true, 1.);
    Trace<ComputationsLog_t, double> select_next_node_calls_trace(
        comp_log, "mean percent of select next node calls", iterations, 20);
    Trace<ComputationsLog_t, double> mean_cost_trace(comp_log, "sol cost mean",
                                                     iterations, 20);
    Trace<ComputationsLog_t, double> stdev_cost_trace(comp_log, "sol cost stdev",
                                                      iterations, 20);
    Timer main_timer;

    vector<double> sol_costs(ants_count);

#ifdef check_state
    vector<uint64_t> state_checks;
#endif

#pragma omp parallel default(shared)
    {
        // Endpoints of new edges (not present in source_route) are inserted
        // into ls_checklist and later used to guide local search
        vector<uint32_t> ls_checklist;
        ls_checklist.reserve(dimension);

        for (int32_t iteration = 0; iteration < iterations; ++iteration) {
#pragma omp barrier
#ifdef check_state
#pragma omp master
            {
                uint64_t hash = 1469598103934665603ULL;
                auto mix = [&](uint64_t x) {
                    hash ^= x;
                    hash *= 1099511628211ULL;
                };
                for (double x : pheromone.trails_) {
                    uint64_t bits;
                    memcpy(&bits, &x, sizeof bits);
                    mix(bits);
                }
                for (const auto &ant : ants)
                    for (auto x : ant.route_)
                        mix(x);
                for (auto x : source_solution->route_)
                    mix(x);
                state_checks.push_back(hash);
            }
#pragma omp barrier
#endif

            if constexpr (use_combined_optimizations) {
                if (iteration == 0) {
// Candidate slots share the same order in all three arrays.
#pragma omp for schedule(static)
                    for (size_t candidate_slot = 0;
                         candidate_slot < nn_product_cache.size(); ++candidate_slot) {
                        nn_product_cache[candidate_slot] =
                            cl_heuristic_cache[candidate_slot] *
                            pheromone.trails_[candidate_slot];
                    }
                }

            } else {
// Load pheromone * heuristic for each edge connecting nearest
// neighbors (up to cl_size)
#pragma omp for schedule(static)
                for (uint32_t node = 0; node < dimension; ++node) {
                    auto cache_it = nn_product_cache.begin() + node * cl_size;
                    auto heuristic_it = cl_heuristic_cache.begin() + node * cl_size;
                    for (auto &nn : problem.get_nearest_neighbors(node, cl_size)) {
                        *cache_it++ = *heuristic_it++ * pheromone.get(node, nn);
                    }
                }
            }

#pragma omp master
            select_next_node_calls = 0;

// Changing schedule from "static" to "dynamic" can speed up
// computations a bit, however it introduces non-determinism due to
// threads scheduling. With "static" the computations always follow
// the same path -- i.e. if we run the program with the same PRNG
// seed (--seed X) then we get exactly the same results.
#pragma omp for schedule(static, 1) reduction(+ : select_next_node_calls)
            for (uint32_t ant_idx = 0; ant_idx < ants.size(); ++ant_idx) {
                uint32_t target_new_edges = opt.min_new_edges_;

                auto &ant = ants[ant_idx];
                ant.initialize(dimension);

                auto start_node = get_rng().next_uint32(dimension);
                ant.visit(start_node);

                ls_checklist.clear();
                ls_checklist.push_back(start_node);

                // We are counting edges (undirected) that are not present in
                // the source_route. The factual # of new edges can be +1 as we
                // skip the check for the closing edge (minor optimization).
                uint32_t new_edges = 0;

                while (ant.visited_count_ < dimension) {
                    auto curr = ant.get_current_node();
                    auto next = select_next_node(
                        pheromone, heuristic,
                        problem.get_nearest_neighbors(curr, cl_size), nn_product_cache,
                        problem.get_backup_neighbors(curr, cl_size, bl_size), ant);
                    ant.visit(next);

                    ++select_next_node_calls;

                    if (!source_solution->contains_edge(curr, next)) {
                        ++new_edges;
                        // The endpoint (tail) of the new edge should be
                        // checked by the local search
                        ls_checklist.push_back(next);
                    }

                    // If we have enough new edges, we try to copy "old" edges
                    // from the source_route.
                    if (new_edges >= target_new_edges) {
                        // Forward direction, start at { next, succ(next) }
                        auto it = source_solution->get_iterator(next);
                        while (ant.try_visit(it.goto_succ())) {
                        }
                        // Backward direction
                        it.goto_pred(); // Reverse .goto_succ() from above
                        while (ant.try_visit(it.goto_pred())) {
                        }
                    }
                }
                if (use_ls) {
                    two_opt_nn(problem, ant.route_, ls_checklist,
                               opt.ls_cand_list_size_);
                }

                ant.cost_ = problem.calculate_route_length(ant.route_);
                sol_costs[ant_idx] = ant.cost_;
            }

#pragma omp master
            {
                iteration_best = &ants.front();
                for (auto &ant : ants) {
                    if (ant.cost_ < iteration_best->cost_) {
                        iteration_best = &ant;
                    }
                }
                if (iteration_best->cost_ < best_ant->cost_) {
                    best_ant->update(iteration_best->route_, iteration_best->cost_);

                    auto error = problem.calc_relative_error(best_ant->cost_);
                    best_cost_trace.add({best_ant->cost_, error}, iteration,
                                        main_timer());

                    model.update_trail_limits(best_ant->cost_);
                }

                auto total_edges = (dimension - 1) * ants_count;
                select_next_node_calls_trace.add(
                    round(100.0 * static_cast<double>(select_next_node_calls) /
                              total_edges,
                          2),
                    iteration, main_timer());

                mean_cost_trace.add(round(sample_mean(sol_costs), 1), iteration);
                stdev_cost_trace.add(round(sample_stdev(sol_costs), 1), iteration);
            }

// Synchronize threads before pheromone update
#pragma omp barrier

            if constexpr (use_combined_optimizations) {
#pragma omp master
                {
                    bool use_best_ant =
                        (get_rng().next_float() < opt.gbest_as_source_prob_);
                    auto &update_ant = use_best_ant ? *best_ant : *iteration_best;
                    source_solution->update(update_ant.route_, update_ant.cost_);
                    const double retention_fraction = 1 - (1 - opt.rho_);
                    pheromone.default_pheromone_value_ =
                        max(pheromone.default_pheromone_value_ * retention_fraction,
                            model.trail_limits_.min_);
                }
#pragma omp barrier
// One writer per row: evaporate, reinforce, then prepare scores.
#pragma omp for schedule(static)
                for (uint32_t city = 0; city < dimension; ++city) {
                    const auto successor_city = source_solution->get_succ(city);
                    const auto predecessor_city = source_solution->get_pred(city);
                    const double pheromone_deposit = 1.0 / source_solution->cost_;
                    const double retention_fraction = 1 - (1 - opt.rho_);
                    for (size_t candidate_slot = size_t(city) * cl_size;
                         candidate_slot < size_t(city + 1) * cl_size;
                         ++candidate_slot) {
                        auto updated_pheromone =
                            max(model.trail_limits_.min_,
                                pheromone.trails_[candidate_slot] * retention_fraction);
                        const auto neighbor = pheromone.nodes_[candidate_slot];
                        if (neighbor == successor_city ||
                            neighbor == predecessor_city) {
                            updated_pheromone =
                                min(model.trail_limits_.max_,
                                    pheromone_deposit + updated_pheromone);
                        }
                        pheromone.trails_[candidate_slot] = updated_pheromone;
                        nn_product_cache[candidate_slot] =
                            cl_heuristic_cache[candidate_slot] * updated_pheromone;
                    }
                }
            } else {
                model.evaporate_pheromone();

#pragma omp master
                {
                    bool use_best_ant =
                        (get_rng().next_float() < opt.gbest_as_source_prob_);
                    auto &update_ant = use_best_ant ? *best_ant : *iteration_best;

                    model.deposit_pheromone(update_ant);

                    // Increase pheromone values on the edges of the new
                    // source_solution
                    source_solution->update(update_ant.route_, update_ant.cost_);
                }
            }
        }
    }
#ifdef check_state
    comp_log("state checks", state_checks);
#endif

    return unique_ptr<Solution>(dynamic_cast<Solution *>(best_ant.release()));
}

int main(int argument_count, char **arguments) {
    try {
        if (argument_count != 8)
            throw runtime_error(
                "Usage: faco baseline|combined instance threads ants iterations seed output.json");
        const string variant = arguments[1];
        if (variant != "baseline" && variant != "combined")
            throw runtime_error("Only baseline and A+B combined are supported");
        ProgramOptions options;
        options.threads_ = stoi(arguments[3]);
        options.ants_count_ = stoul(arguments[4]);
        options.iterations_ = stoi(arguments[5]);
        options.seed_ = stoull(arguments[6]);
        if (options.threads_ < 1 || options.threads_ > 64 || options.ants_count_ < 2 ||
            options.iterations_ < 1 || options.seed_ == 0)
            throw runtime_error("Invalid FACO experiment parameters");
        omp_set_dynamic(0);
        omp_set_num_threads(options.threads_);
        init_random_number_generators(options.seed_);
        auto problem = load_tsplib_instance(arguments[2]);
        if (!problem.is_symmetric_ || problem.edge_weight_type_ != EUC_2D ||
            problem.dimension_ < 81)
            throw runtime_error(
                "FACO requires symmetric EUC_2D with at least 81 cities");
        using json = nlohmann::json;
        using Log = ComputationsLog<json>;
        json record, execution;
        Timer preprocessing_timer;
        problem.compute_nn_lists(80);
        record["nn and backup lists calc time"] = preprocessing_timer();
        Log computation_log(execution, cout);
        Timer solver_timer;
        auto result = variant == "baseline"
                          ? run_focused_aco<false>(problem, options, computation_log)
                          : run_focused_aco<true>(problem, options, computation_log);
        const double solver_seconds = solver_timer();
        if (!problem.is_route_valid(result->route_) ||
            problem.calculate_route_length(result->route_) != result->cost_)
            throw runtime_error("FACO returned an invalid tour");
        execution["execution time"] = solver_seconds;
        execution["final route"] = result->route_;
        execution["final cost"] = result->cost_;
        record["variant"] = variant;
        record["executions"] = json::array({execution});
        dump(options, record["args"]);
        ofstream output(arguments[7]);
        output << record.dump(2) << '\n';
        output.close();
        if (!output)
            throw runtime_error("Cannot write FACO result");
        cout << variant << ": cost " << result->cost_ << ", solver seconds "
             << solver_seconds << '\n';
        return 0;
    } catch (const exception &error) {
        cerr << error.what() << '\n';
        return 1;
    }
}


## 4. ILS: normal versus focused repair and selective sharing
`ils` performs candidate-based 2-opt refinement starting from all cities and keeps
eight trajectories independent. `cooperate` repairs changed-edge endpoints and
periodically copies the best tour to one stalled trajectory. Initialization,
perturbations, seeds, thread counts and budgets are otherwise shared. This is a
parallel normal-ILS baseline, not a single sequential trajectory.


In [ ]:
%%writefile /content/parallel_tsp_cpp/focused_sharing.cpp
// Non-ACO experiment; reuses the pinned author's TSP parser and 2/3-opt routines.
#include <bits/stdc++.h>
#include <omp.h>
#include "problem_instance.h"
#include "local_search.h"
#include "rand.h"
#include "json.hpp"

using namespace std;

using Tour = vector<uint32_t>;
using json = nlohmann::json;
constexpr int search_trajectory_count = 8;
constexpr uint32_t candidate_neighbor_count = 20;

Tour nearest_tour(const ProblemInstance &problem_instance, uint32_t start_city) {
    Tour tour;
    vector<char> visited_cities(problem_instance.dimension_, false);
    uint32_t current_city = start_city;
    while (tour.size() < problem_instance.dimension_) {
        tour.push_back(current_city);
        visited_cities[current_city] = true;
        if (tour.size() == problem_instance.dimension_)
            break;
        uint32_t next_city = problem_instance.dimension_;
        for (auto candidate_city : problem_instance.get_nearest_neighbors(
                 current_city, candidate_neighbor_count)) {
            if (!visited_cities[candidate_city]) {
                next_city = candidate_city;
                break;
            }
        }
        if (next_city == problem_instance.dimension_) {
            double nearest_distance = numeric_limits<double>::infinity();
            for (uint32_t candidate_city = 0;
                 candidate_city < problem_instance.dimension_; ++candidate_city) {
                if (!visited_cities[candidate_city] &&
                    problem_instance.get_distance(current_city, candidate_city) <
                        nearest_distance) {
                    next_city = candidate_city;
                    nearest_distance =
                        problem_instance.get_distance(current_city, candidate_city);
                }
            }
        }
        current_city = next_city;
    }
    return tour;
}

void kick(Tour &tour, Random &random_generator, Tour &affected_cities) {
    const uint32_t city_count = tour.size(), segment_span = city_count / 4;
    const uint32_t rotation_offset = random_generator.next_uint32(city_count);
    rotate(tour.begin(), tour.begin() + rotation_offset, tour.end());
    const uint32_t first_cut_position = 1 + random_generator.next_uint32(segment_span);
    const uint32_t second_cut_position =
        first_cut_position + 1 + random_generator.next_uint32(segment_span);
    const uint32_t third_cut_position =
        second_cut_position + 1 + random_generator.next_uint32(segment_span);
    Tour perturbed_tour;
    perturbed_tour.reserve(city_count);
    // A B C D -> A D C B changes four boundary connections, retaining interiors.
    for (auto segment :
         {pair{0u, first_cut_position}, pair{third_cut_position, city_count},
          pair{second_cut_position, third_cut_position},
          pair{first_cut_position, second_cut_position}}) {
        affected_cities.push_back(tour[segment.first]);
        affected_cities.push_back(tour[segment.second - 1]);
        perturbed_tour.insert(perturbed_tour.end(), tour.begin() + segment.first,
                              tour.begin() + segment.second);
    }
    tour.swap(perturbed_tour);
}

struct Walker {
    Random random_generator;
    Tour tour;
    double tour_length = numeric_limits<double>::infinity();
    uint32_t stagnation_attempts = 0;
    uint64_t completed_jobs = 0;
    bool improved_this_round = false;
    double accepted_at_seconds = 0;
};

void self_test() {
    Random random_generator;
    random_generator.init(19);
    vector<Vec2d> city_coordinates;
    for (int city_index = 0; city_index < 96; ++city_index)
        city_coordinates.push_back({double(random_generator.next_uint32(1000)),
                                    double(random_generator.next_uint32(1000))});
    city_coordinates[1] = city_coordinates[0];
    ProblemInstance problem_instance(96, EUC_2D, city_coordinates, {}, true);
    problem_instance.compute_nn_lists(candidate_neighbor_count);
    auto tour = nearest_tour(problem_instance, 0);
    assert(problem_instance.is_route_valid(tour));
    two_opt_nn(problem_instance, tour, true, candidate_neighbor_count);
    for (int iteration_index = 0; iteration_index < 200; ++iteration_index) {
        Tour changed_cities;
        kick(tour, random_generator, changed_cities);
        assert(problem_instance.is_route_valid(tour));
        double length_before_repair = problem_instance.calculate_route_length(tour);
        if (iteration_index % 2)
            two_opt_nn(problem_instance, tour, changed_cities,
                       candidate_neighbor_count);
        else
            two_opt_nn(problem_instance, tour, true, candidate_neighbor_count);
        assert(problem_instance.is_route_valid(tour));
        assert(problem_instance.calculate_route_length(tour) <= length_before_repair);
    }
    cout
        << "Permutation, duplicate-coordinate, perturbation and monotone refinement checks passed\n";
}

int main(int argument_count, char **arguments) {
    try {
        if (argument_count == 2 && string(arguments[1]) == "--self-test") {
            self_test();
            return 0;
        }
        if (argument_count != 8)
            throw runtime_error(
                "Usage: solver instance mode threads seed seconds rounds output.json");
        string search_mode = arguments[2];
        int thread_count = stoi(arguments[3]), round_limit = stoi(arguments[6]);
        uint64_t random_seed = stoull(arguments[4]);
        double time_budget_seconds = stod(arguments[5]);
        if ((search_mode != "ils" && search_mode != "cooperate") || thread_count < 1 ||
            thread_count > 64 || !isfinite(time_budget_seconds) ||
            time_budget_seconds < 0 || round_limit < 0 ||
            (time_budget_seconds == 0 && round_limit == 0))
            throw runtime_error("Invalid experiment parameters");
        omp_set_dynamic(0);
        omp_set_num_threads(thread_count);
        const double search_start_time = omp_get_wtime();
        auto problem_instance = load_tsplib_instance(arguments[1]);
        if (!problem_instance.is_symmetric_ ||
            problem_instance.edge_weight_type_ != EUC_2D ||
            problem_instance.dimension_ < 24)
            throw runtime_error(
                "Experiment requires EUC_2D symmetric TSP, at least 24 cities");
        problem_instance.compute_nn_lists(candidate_neighbor_count);
        const double preprocessing_seconds = omp_get_wtime() - search_start_time;
        vector<Walker> walkers(search_trajectory_count);
        for (int worker_id = 0; worker_id < search_trajectory_count; ++worker_id)
            walkers[worker_id].random_generator.init(
                random_seed + UINT64_C(0x9e3779b97f4a7c15) * uint64_t(worker_id + 1));
        Tour best_tour;
        double best_tour_length = numeric_limits<double>::infinity();
        json improvement_history = json::array();
        int completed_rounds = 0, elite_transfer_count = 0;
        const double deadline_time = time_budget_seconds > 0
                                         ? search_start_time + time_budget_seconds
                                         : numeric_limits<double>::infinity();
        while ((round_limit == 0 || completed_rounds < round_limit) &&
               omp_get_wtime() < deadline_time) {
#pragma omp parallel for schedule(dynamic, 1)
            for (int worker_id = 0; worker_id < search_trajectory_count; ++worker_id) {
                auto &walker = walkers[worker_id];
                walker.improved_this_round = false;
                if (omp_get_wtime() >= deadline_time)
                    continue;
                Tour candidate_tour, changed_cities;
                bool requires_initial_tour = walker.tour.empty();
                if (requires_initial_tour) {
                    candidate_tour = nearest_tour(problem_instance,
                                                  walker.random_generator.next_uint32(
                                                      problem_instance.dimension_));
                    two_opt_nn(problem_instance, candidate_tour, true,
                               candidate_neighbor_count);
                    three_opt_nn(problem_instance, candidate_tour, true,
                                 candidate_neighbor_count);
                } else {
                    candidate_tour = walker.tour;
                    kick(candidate_tour, walker.random_generator, changed_cities);
                    if (walker.stagnation_attempts >= 100)
                        kick(candidate_tour, walker.random_generator, changed_cities);
                    if (search_mode == "ils")
                        two_opt_nn(problem_instance, candidate_tour, true,
                                   candidate_neighbor_count);
                    else {
                        // shortcut: candidate checklist may miss distant gains;
                        // benchmark against full scans.
                        two_opt_nn(problem_instance, candidate_tour, changed_cities,
                                   candidate_neighbor_count);
                    }
                }
                double tour_length =
                    problem_instance.calculate_route_length(candidate_tour);
                double completion_time = omp_get_wtime();
                ++walker.completed_jobs;
                // Ignore late results; unfinished refinement is allowed to finish
                // safely.
                if (completion_time > deadline_time)
                    continue;
                if (tour_length <= walker.tour_length) {
                    walker.improved_this_round = tour_length < walker.tour_length;
                    walker.stagnation_attempts =
                        walker.improved_this_round ? 0 : walker.stagnation_attempts + 1;
                    walker.tour.swap(candidate_tour);
                    walker.tour_length = tour_length;
                    walker.accepted_at_seconds = completion_time - search_start_time;
                } else
                    ++walker.stagnation_attempts;
            }
            vector<int> completion_order(search_trajectory_count);
            iota(completion_order.begin(), completion_order.end(), 0);
            sort(completion_order.begin(), completion_order.end(),
                 [&](int left_worker, int right_worker) {
                     return walkers[left_worker].accepted_at_seconds <
                                walkers[right_worker].accepted_at_seconds ||
                            (walkers[left_worker].accepted_at_seconds ==
                                 walkers[right_worker].accepted_at_seconds &&
                             left_worker < right_worker);
                 });
            for (int worker_id : completion_order) {
                const auto &walker = walkers[worker_id];
                if (walker.improved_this_round &&
                    walker.tour_length < best_tour_length) {
                    best_tour_length = walker.tour_length;
                    best_tour = walker.tour;
                    improvement_history.push_back(
                        {{"seconds", walker.accepted_at_seconds},
                         {"cost", best_tour_length}});
                }
            }
            ++completed_rounds;
            if (search_mode == "cooperate" && completed_rounds % 32 == 0 &&
                !best_tour.empty()) {
                auto worst_walker = max_element(
                    walkers.begin(), walkers.end(),
                    [](const auto &left_worker, const auto &right_worker) {
                        return left_worker.tour_length < right_worker.tour_length;
                    });
                if (worst_walker->stagnation_attempts >= 32 &&
                    worst_walker->tour_length > best_tour_length) {
                    // Replace only one stalled trajectory, leaving the other seven
                    // independent.
                    worst_walker->tour = best_tour;
                    worst_walker->tour_length = best_tour_length;
                    worst_walker->stagnation_attempts = 0;
                    ++elite_transfer_count;
                }
            }
        }
        const double elapsed_seconds = omp_get_wtime() - search_start_time;
        if (best_tour.empty())
            throw runtime_error("Budget expired before any tour completed");
        if (!problem_instance.is_route_valid(best_tour) ||
            problem_instance.calculate_route_length(best_tour) != best_tour_length)
            throw runtime_error("Invalid final tour");
        json walker_states = json::array();
        uint64_t completed_jobs = 0;
        for (const auto &walker : walkers) {
            walker_states.push_back({{"cost", walker.tour_length},
                                     {"route", walker.tour},
                                     {"jobs", walker.completed_jobs}});
            completed_jobs += walker.completed_jobs;
        }
        json result_data = {{"mode", search_mode},
                            {"threads", thread_count},
                            {"seed", random_seed},
                            {"budget_seconds", time_budget_seconds},
                            {"preprocessing_seconds", preprocessing_seconds},
                            {"elapsed_seconds", elapsed_seconds},
                            {"rounds", completed_rounds},
                            {"jobs", completed_jobs},
                            {"migrations", elite_transfer_count},
                            {"cost", best_tour_length},
                            {"route", best_tour},
                            {"trace", improvement_history},
                            {"walkers", walker_states}};
        ofstream result_file(arguments[7]);
        if (!result_file)
            throw runtime_error("Cannot open result path");
        result_file << result_data.dump();
        result_file.close();
        if (!result_file)
            throw runtime_error("Cannot write result");
        cout << "Final cost " << best_tour_length << ", jobs " << completed_jobs
             << ", seconds " << elapsed_seconds << "\n";
        return 0;
    } catch (const exception &error) {
        cerr << error.what() << "\n";
        return 1;
    }
}


## 5. C++ process runner and independent validation
Every output tour must contain each city exactly once. Its distance is recomputed
independently from coordinates. Process failures and timeouts stop the experiment;
fresh result directories prevent old outputs from being mistaken for new results.


In [ ]:
%%writefile /content/parallel_tsp_cpp/experiment_support.hpp
#pragma once

#include <bits/stdc++.h>
#include "json.hpp"

#ifdef _WIN32
#ifndef NOMINMAX
#define NOMINMAX
#endif
#include <windows.h>
#else
#include <fcntl.h>
#include <signal.h>
#include <spawn.h>
#include <sys/wait.h>
#include <unistd.h>
extern char **environ;
#endif

using namespace std;
using json = nlohmann::json;
namespace fs = filesystem;
using Clock = chrono::steady_clock;

double seconds_since(Clock::time_point start) {
    return chrono::duration<double>(Clock::now() - start).count();
}

void require(bool condition, const string &message) {
    if (!condition)
        throw runtime_error(message);
}

void write_text(const fs::path &path, const string &text) {
    ofstream output(path);
    output << text;
    output.close();
    require(bool(output), "Cannot write " + path.string());
}

json read_json(const fs::path &path) {
    ifstream input(path);
    require(bool(input), "Cannot open " + path.string());
    json value;
    input >> value;
    return value;
}

fs::path executable(const string &name) {
#ifdef _WIN32
    return fs::absolute(name + ".exe");
#else
    return fs::absolute(name);
#endif
}

#ifdef _WIN32
string quote_argument(const string &argument) {
    string quoted = "\"";
    size_t backslashes = 0;
    for (char character : argument) {
        if (character == '\\') {
            ++backslashes;
            continue;
        }
        quoted.append(backslashes * (character == '"' ? 2 : 1), '\\');
        backslashes = 0;
        if (character == '"')
            quoted += '\\';
        quoted += character;
    }
    quoted.append(backslashes * 2, '\\');
    return quoted + '"';
}
#endif

// Arguments go directly to the process API, never through a shell.
double run_process(const vector<string> &arguments, const fs::path &log_path,
                   double timeout_seconds = 600) {
    require(!arguments.empty() && timeout_seconds > 0, "Invalid process request");
    const auto start = Clock::now();
#ifdef _WIN32
    SECURITY_ATTRIBUTES security = {sizeof(SECURITY_ATTRIBUTES), nullptr, TRUE};
    HANDLE log_file =
        CreateFileW(log_path.wstring().c_str(), GENERIC_WRITE, FILE_SHARE_READ,
                    &security, CREATE_ALWAYS, FILE_ATTRIBUTE_NORMAL, nullptr);
    require(log_file != INVALID_HANDLE_VALUE, "Cannot open process log");
    string command;
    for (const auto &argument : arguments)
        command += quote_argument(argument) + ' ';
    STARTUPINFOA startup = {};
    startup.cb = sizeof(startup);
    startup.dwFlags = STARTF_USESTDHANDLES;
    startup.hStdOutput = log_file;
    startup.hStdError = log_file;
    startup.hStdInput = GetStdHandle(STD_INPUT_HANDLE);
    PROCESS_INFORMATION process = {};
    const bool created =
        CreateProcessA(nullptr, command.data(), nullptr, nullptr, TRUE,
                       CREATE_NO_WINDOW, nullptr, nullptr, &startup, &process);
    CloseHandle(log_file);
    require(created, "Cannot start " + arguments.front());
    const auto wait_result = WaitForSingleObject(
        process.hProcess, static_cast<DWORD>(timeout_seconds * 1000));
    if (wait_result != WAIT_OBJECT_0) {
        TerminateProcess(process.hProcess, 1);
        WaitForSingleObject(process.hProcess, INFINITE);
    }
    DWORD exit_code = 1;
    GetExitCodeProcess(process.hProcess, &exit_code);
    CloseHandle(process.hThread);
    CloseHandle(process.hProcess);
    require(wait_result == WAIT_OBJECT_0,
            "Process timed out; see " + log_path.string());
    require(exit_code == 0, "Process failed; see " + log_path.string());
#else
    vector<char *> argv;
    for (const auto &argument : arguments)
        argv.push_back(const_cast<char *>(argument.c_str()));
    argv.push_back(nullptr);
    posix_spawn_file_actions_t actions;
    require(posix_spawn_file_actions_init(&actions) == 0, "Cannot prepare process");
    const int open_status = posix_spawn_file_actions_addopen(
        &actions, STDOUT_FILENO, log_path.c_str(), O_WRONLY | O_CREAT | O_TRUNC, 0600);
    const int redirect_status =
        posix_spawn_file_actions_adddup2(&actions, STDOUT_FILENO, STDERR_FILENO);
    pid_t process_id = -1;
    const int spawn_status = (open_status || redirect_status)
                                 ? EINVAL
                                 : posix_spawn(&process_id, argv[0], &actions, nullptr,
                                               argv.data(), environ);
    posix_spawn_file_actions_destroy(&actions);
    require(spawn_status == 0, "Cannot start " + arguments.front() + ": " +
                                   string(strerror(spawn_status)));
    int status = 0;
    while (true) {
        const auto wait_result = waitpid(process_id, &status, WNOHANG);
        if (wait_result == process_id)
            break;
        if (wait_result < 0 && errno == EINTR)
            continue;
        if (wait_result < 0 || seconds_since(start) > timeout_seconds) {
            kill(process_id, SIGKILL);
            while (waitpid(process_id, &status, 0) < 0 && errno == EINTR) {
            }
            throw runtime_error("Process failed or timed out; see " +
                                log_path.string());
        }
        this_thread::sleep_for(chrono::milliseconds(2));
    }
    require(WIFEXITED(status) && WEXITSTATUS(status) == 0,
            "Process failed; see " + log_path.string());
#endif
    return seconds_since(start);
}

string trim(string text) {
    const auto first = text.find_first_not_of(" \t\r\n");
    if (first == string::npos)
        return "";
    return text.substr(first, text.find_last_not_of(" \t\r\n") - first + 1);
}

using Coordinates = vector<pair<double, double>>;

Coordinates read_coordinates(const fs::path &path) {
    ifstream input(path);
    require(bool(input), "Cannot open instance: " + path.string());
    map<string, string> header;
    Coordinates coordinates;
    string line;
    bool reading_coordinates = false;
    while (getline(input, line)) {
        line = trim(line);
        if (line.empty())
            continue;
        if (line == "NODE_COORD_SECTION") {
            reading_coordinates = true;
            continue;
        }
        if (line == "EOF")
            break;
        if (!reading_coordinates) {
            const auto separator = line.find(':');
            if (separator != string::npos)
                header[trim(line.substr(0, separator))] =
                    trim(line.substr(separator + 1));
            continue;
        }
        istringstream row(line);
        int city_id;
        double x_coordinate, y_coordinate;
        string extra;
        require(bool(row >> city_id >> x_coordinate >> y_coordinate) && !(row >> extra),
                "Invalid coordinate row");
        require(city_id == static_cast<int>(coordinates.size()) + 1 &&
                    isfinite(x_coordinate) && isfinite(y_coordinate),
                "Expected ordered city IDs and finite coordinates");
        coordinates.emplace_back(x_coordinate, y_coordinate);
    }
    require(header["TYPE"] == "TSP" && header["EDGE_WEIGHT_TYPE"] == "EUC_2D",
            "Only symmetric EUC_2D TSP instances are supported");
    require(header.count("DIMENSION") && !coordinates.empty() &&
                coordinates.size() == stoull(header.at("DIMENSION")),
            "Coordinate count does not match DIMENSION");
    return coordinates;
}

long long validate_route(const Coordinates &coordinates, const json &route,
                         double expected_cost) {
    require(route.is_array() && route.size() == coordinates.size(),
            "Wrong number of cities in route");
    vector<bool> visited(coordinates.size(), false);
    vector<size_t> cities;
    for (const auto &value : route) {
        require(value.is_number_integer(), "Route city must be an integer");
        const auto city = value.get<long long>();
        require(city >= 0 && static_cast<size_t>(city) < coordinates.size() &&
                    !visited[city],
                "Duplicate or out-of-range city");
        visited[city] = true;
        cities.push_back(static_cast<size_t>(city));
    }
    long long cost = 0;
    for (size_t position = 0; position < cities.size(); ++position) {
        const auto [from_x, from_y] = coordinates[cities[position]];
        const auto [to_x, to_y] = coordinates[cities[(position + 1) % cities.size()]];
        const double distance = floor(hypot(from_x - to_x, from_y - to_y) + 0.5);
        require(isfinite(distance) && distance >= 0 &&
                    distance <
                        static_cast<double>(numeric_limits<long long>::max() - cost),
                "Route cost exceeds supported range");
        cost += static_cast<long long>(distance);
    }
    require(isfinite(expected_cost) && static_cast<double>(cost) == expected_cost,
            "Reported tour cost does not match independent calculation");
    return cost;
}

double median(vector<double> values) {
    require(!values.empty(), "Cannot summarize empty results");
    sort(values.begin(), values.end());
    const auto middle = values.size() / 2;
    return values.size() % 2 ? values[middle]
                             : (values[middle - 1] + values[middle]) / 2;
}

fs::path new_result_directory(const string &suite) {
    fs::create_directories("results");
    const auto timestamp = chrono::duration_cast<chrono::microseconds>(
                               chrono::system_clock::now().time_since_epoch())
                               .count();
    const fs::path directory =
        fs::path("results") / (suite + "_" + to_string(timestamp));
    require(fs::create_directory(directory), "Result directory already exists");
    return directory;
}

void validate_trace(const json &result, double budget_seconds) {
    double previous_cost = numeric_limits<double>::infinity();
    double previous_seconds = 0;
    for (const auto &event : result.at("trace")) {
        const double cost = event.at("cost");
        const double seconds = event.at("seconds");
        require(isfinite(cost) && isfinite(seconds) && cost < previous_cost &&
                    seconds >= previous_seconds &&
                    (budget_seconds == 0 || seconds <= budget_seconds),
                "Invalid ILS improvement trace");
        previous_cost = cost;
        previous_seconds = seconds;
    }
    require(previous_cost == result.at("cost").get<double>(),
            "Final cost does not match improvement trace");
}


## 6. C++ benchmark and comparison runner
Runs paired seeds in shuffled order and writes raw results and comma-separated
values (CSV) tables. FACO pairs must have matching routes and quality traces.
ILS methods are expected to differ in quality; positive paired cost improvement
means focused repair and selective sharing found a shorter tour.


In [ ]:
%%writefile /content/parallel_tsp_cpp/experiments.cpp
#include <bits/stdc++.h>
#include <omp.h>
#include "experiment_support.hpp"
#include "settings.hpp"

using namespace std;

json run_faco(const ExperimentSettings &settings, const Coordinates &coordinates,
              const fs::path &instance, const fs::path &directory,
              const string &variant, int seed) {
    const auto prefix = directory / (variant + "_seed_" + to_string(seed));
    const string output_path = prefix.string() + ".json";
    const double process_seconds =
        run_process({executable("faco").string(), variant, instance.string(),
                     to_string(settings.thread_count), to_string(settings.ant_count),
                     to_string(settings.faco_iterations), to_string(seed), output_path},
                    prefix.string() + ".log");
    auto result = read_json(output_path);
    const auto &execution = result.at("executions").at(0);
    validate_route(coordinates, execution.at("final route"),
                   execution.at("final cost"));
    result["process_seconds"] = process_seconds;
    write_text(output_path, result.dump(2));
    return result;
}

void compare_faco(const ExperimentSettings &settings, const Coordinates &coordinates,
                  const fs::path &instance, const fs::path &directory) {
    require(coordinates.size() >= 81, "FACO needs at least 81 cities");
    vector<double> baseline_solver_times, combined_solver_times;
    vector<double> baseline_process_times, combined_process_times;
    vector<double> solver_speedups, process_speedups, costs;
    ostringstream rows;
    rows << setprecision(12)
         << "variant,seed,cost,solver_seconds,process_seconds,preprocessing_seconds\n";
    mt19937 order_generator(20261031);
    for (const int seed : settings.seeds) {
        vector<string> order = {"baseline", "combined"};
        shuffle(order.begin(), order.end(), order_generator);
        map<string, json> paired_results;
        for (const auto &variant : order) {
            auto result =
                run_faco(settings, coordinates, instance, directory, variant, seed);
            const auto &execution = result.at("executions").at(0);
            const double cost = execution.at("final cost");
            const double solver_seconds = execution.at("execution time");
            const double process_seconds = result.at("process_seconds");
            rows << variant << ',' << seed << ',' << cost << ',' << solver_seconds
                 << ',' << process_seconds << ','
                 << result.at("nn and backup lists calc time").get<double>() << '\n';
            write_text(directory / "runs.csv", rows.str());
            cout << variant << ", seed " << seed << ": cost " << cost << ", solver "
                 << solver_seconds << " s, process " << process_seconds << " s" << endl;
            paired_results.emplace(variant, move(result));
        }
        const auto &baseline = paired_results.at("baseline").at("executions").at(0);
        const auto &combined = paired_results.at("combined").at("executions").at(0);
        for (const string field : {"final route", "final cost", "best sol cost values",
                                   "sol cost mean values", "sol cost stdev values"})
            require(baseline.at(field) == combined.at(field),
                    "FACO paired results differ: " + field);
        if (baseline.count("state checks") || combined.count("state checks"))
            require(baseline.at("state checks") == combined.at("state checks"),
                    "FACO internal states differ");
        const double baseline_solver = baseline.at("execution time");
        const double combined_solver = combined.at("execution time");
        const double baseline_process =
            paired_results.at("baseline").at("process_seconds");
        const double combined_process =
            paired_results.at("combined").at("process_seconds");
        require(combined_solver > 0 && combined_process > 0, "Invalid measured time");
        baseline_solver_times.push_back(baseline_solver);
        combined_solver_times.push_back(combined_solver);
        baseline_process_times.push_back(baseline_process);
        combined_process_times.push_back(combined_process);
        solver_speedups.push_back(baseline_solver / combined_solver);
        process_speedups.push_back(baseline_process / combined_process);
        costs.push_back(baseline.at("final cost"));
    }
    ostringstream summary;
    summary << setprecision(12)
            << "method,median_solver_seconds,median_process_seconds,mean_cost\n";
    const double mean_cost = accumulate(costs.begin(), costs.end(), 0.0) / costs.size();
    summary << "FACO baseline," << median(baseline_solver_times) << ','
            << median(baseline_process_times) << ',' << mean_cost << '\n';
    summary << "FACO A+B," << median(combined_solver_times) << ','
            << median(combined_process_times) << ',' << mean_cost << '\n';
    write_text(directory / "summary.csv", summary.str());
    ostringstream speedups;
    speedups
        << setprecision(12)
        << "comparison,median_paired_solver_speedup,median_paired_process_speedup\n"
        << "Baseline vs A+B," << median(solver_speedups) << ','
        << median(process_speedups) << '\n';
    write_text(directory / "paired_speedups.csv", speedups.str());
    cout << '\n'
         << summary.str() << speedups.str()
         << "All paired FACO routes and quality traces match.\n";
    write_text(directory / "validation.json",
               json({{"paired_routes_and_quality_traces_match", true},
                     {"paired_seeds", settings.seeds}})
                   .dump(2));
}

json run_ils(const Coordinates &coordinates, const fs::path &instance,
             const fs::path &directory, const string &mode, int threads, int seed,
             double seconds, int rounds, const string &label) {
    const auto result_path = directory / (label + ".json");
    const double process_seconds =
        run_process({executable("focused_sharing").string(), instance.string(), mode,
                     to_string(threads), to_string(seed), to_string(seconds),
                     to_string(rounds), result_path.string()},
                    directory / (label + ".log"), max(300.0, seconds * 3));
    auto result = read_json(result_path);
    validate_route(coordinates, result.at("route"), result.at("cost"));
    validate_trace(result, seconds);
    require(result.at("mode") == mode && result.at("threads") == threads &&
                result.at("seed") == seed,
            "Unexpected ILS run metadata");
    result["process_seconds"] = process_seconds;
    write_text(result_path, result.dump(2));
    return result;
}

void compare_ils(const ExperimentSettings &settings, const Coordinates &coordinates,
                 const fs::path &instance, const fs::path &directory) {
    map<string, vector<double>> costs, process_times, search_times;
    vector<double> cost_improvements, elapsed_ratios;
    ostringstream rows, pairs;
    rows
        << setprecision(12)
        << "method,seed,cost,budget_seconds,solver_seconds,process_seconds,jobs,migrations\n";
    pairs << setprecision(12)
          << "seed,cost_improvement_percent,normal_over_enhanced_process_time\n";
    mt19937 order_generator(20261031);
    for (const int seed : settings.seeds) {
        vector<string> order = {"ils", "cooperate"};
        shuffle(order.begin(), order.end(), order_generator);
        map<string, json> results;
        for (const auto &mode : order) {
            auto result = run_ils(coordinates, instance, directory, mode,
                                  settings.thread_count, seed, settings.ils_seconds, 0,
                                  mode + "_seed_" + to_string(seed));
            const double cost = result.at("cost");
            const double process_seconds = result.at("process_seconds");
            const double solver_seconds = result.at("elapsed_seconds");
            costs[mode].push_back(cost);
            process_times[mode].push_back(process_seconds);
            search_times[mode].push_back(solver_seconds);
            rows << mode << ',' << seed << ',' << cost << ',' << settings.ils_seconds
                 << ',' << solver_seconds << ',' << process_seconds << ','
                 << result.at("jobs") << ',' << result.at("migrations") << '\n';
            write_text(directory / "runs.csv", rows.str());
            cout << mode << ", seed " << seed << ": cost " << cost << ", process "
                 << process_seconds << " s, jobs " << result.at("jobs")
                 << ", transfers " << result.at("migrations") << endl;
            results.emplace(mode, move(result));
        }
        const double normal_cost = results.at("ils").at("cost");
        const double enhanced_cost = results.at("cooperate").at("cost");
        // Zero-length instances have no percentage improvement to calculate.
        const double improvement =
            normal_cost > 0 ? 100 * (normal_cost - enhanced_cost) / normal_cost : 0;
        const double elapsed_ratio =
            results.at("ils").at("process_seconds").get<double>() /
            results.at("cooperate").at("process_seconds").get<double>();
        cost_improvements.push_back(improvement);
        elapsed_ratios.push_back(elapsed_ratio);
        pairs << seed << ',' << improvement << ',' << elapsed_ratio << '\n';
        write_text(directory / "paired_comparison.csv", pairs.str());
    }
    ostringstream summary;
    summary
        << setprecision(12)
        << "method,mean_cost,best_cost,median_solver_seconds,median_process_seconds\n";
    for (const string mode : {"ils", "cooperate"}) {
        const string label =
            mode == "ils" ? "Normal ILS" : "ILS + focused repair + selective sharing";
        summary << label << ','
                << accumulate(costs[mode].begin(), costs[mode].end(), 0.0) /
                       costs[mode].size()
                << ',' << *min_element(costs[mode].begin(), costs[mode].end()) << ','
                << median(search_times[mode]) << ',' << median(process_times[mode])
                << '\n';
    }
    write_text(directory / "summary.csv", summary.str());
    cout
        << '\n'
        << summary.str()
        << "Median paired cost improvement (%): " << median(cost_improvements)
        << "\nPositive means focused repair + sharing found shorter tours.\n"
        << "Median normal/enhanced process-time ratio: " << median(elapsed_ratios)
        << "\nEqual time budgets compare solution quality, not time to the same solution.\n";
}

void run_checks(const ExperimentSettings &settings, const Coordinates &coordinates,
                const fs::path &instance, const fs::path &directory) {
    const Coordinates square = {{0, 0}, {3, 0}, {3, 4}, {0, 4}};
    require(validate_route(square, json::array({0, 1, 2, 3}), 14) == 14,
            "Independent distance validation failed");
    for (const auto &bad_route : {json::array({0, 1, 1, 3}), json::array({0, 1, 2, 4}),
                                  json::array({0, 1, 2}), json::array({0, 1, 2, -1})}) {
        bool rejected = false;
        try {
            validate_route(square, bad_route, 14);
        } catch (const exception &) {
            rejected = true;
        }
        require(rejected, "Invalid tour was accepted");
    }
    require(median({4, 1, 2, 3}) == 2.5 && median({3, 1, 2}) == 2,
            "Median calculation failed");
    run_process({executable("focused_sharing").string(), "--self-test"},
                directory / "solver_self_test.log");
    for (const string mode : {"ils", "cooperate"}) {
        const auto sequential = run_ils(coordinates, instance, directory, mode, 1, 101,
                                        0, settings.check_rounds, mode + "_one_thread");
        const auto parallel =
            run_ils(coordinates, instance, directory, mode, settings.thread_count, 101,
                    0, settings.check_rounds, mode + "_parallel");
        for (const string field :
             {"route", "cost", "walkers", "migrations", "jobs", "rounds"})
            require(sequential.at(field) == parallel.at(field),
                    "ILS thread-count check failed: " + mode + " " + field);
        cout << mode << ": fixed-round states match across thread counts; transfers "
             << parallel.at("migrations") << endl;
    }
    write_text(directory / "validation.json",
               json({{"independent_validator_checks", true},
                     {"solver_self_test", true},
                     {"fixed_round_thread_states_match", true},
                     {"rounds", settings.check_rounds}})
                   .dump(2));
}

int main(int argument_count, char **arguments) {
    try {
        require(argument_count >= 2 && argument_count <= 3,
                "Usage: experiments faco|ils|check [--smoke]");
        const string suite = arguments[1];
        require(suite == "faco" || suite == "ils" || suite == "check", "Unknown suite");
        ExperimentSettings settings;
        if (argument_count == 3) {
            require(string(arguments[2]) == "--smoke", "Unknown option");
            settings.thread_count = min(2, omp_get_num_procs());
            settings.seeds = {11, 23};
            settings.faco_iterations = 300;
            settings.ils_seconds = 2;
        }
        require(settings.thread_count >= 1 && settings.thread_count <= 64 &&
                    settings.faco_iterations > 0 && settings.ant_count >= 2 &&
                    isfinite(settings.ils_seconds) && settings.ils_seconds > 0 &&
                    settings.check_rounds > 0 && !settings.seeds.empty(),
                "Invalid experiment settings");
        require(set<int>(settings.seeds.begin(), settings.seeds.end()).size() ==
                        settings.seeds.size() &&
                    *min_element(settings.seeds.begin(), settings.seeds.end()) > 0,
                "Use unique positive seeds");
#ifdef _WIN32
        _putenv_s("OMP_NUM_THREADS", to_string(settings.thread_count).c_str());
        _putenv_s("OMP_DYNAMIC", "FALSE");
        _putenv_s("OMP_WAIT_POLICY", "ACTIVE");
#else
        setenv("OMP_NUM_THREADS", to_string(settings.thread_count).c_str(), 1);
        setenv("OMP_DYNAMIC", "FALSE", 1);
        setenv("OMP_WAIT_POLICY", "ACTIVE", 1);
#endif
        const auto instance = fs::absolute(fs::path("deps/faco/instances") /
                                           (settings.instance_name + ".tsp"));
        const auto coordinates = read_coordinates(instance);
        require(coordinates.size() >= 24, "ILS needs at least 24 cities");
        const auto directory = new_result_directory(suite);
        const json metadata = {
            {"suite", suite},
            {"instance", instance.string()},
            {"cities", coordinates.size()},
            {"threads", settings.thread_count},
            {"seeds", settings.seeds},
            {"ants", settings.ant_count},
            {"faco_iterations", settings.faco_iterations},
            {"ils_seconds", settings.ils_seconds},
            {"check_rounds", settings.check_rounds},
            {"smoke_test", argument_count == 3},
            {"faco_revision", "05b7c1b8d52b27d121b6138222f986fcbf196863"}};
        write_text(directory / "settings.json", metadata.dump(2));
        cout << "Instance " << settings.instance_name << ": " << coordinates.size()
             << " cities, threads " << settings.thread_count << endl;
        if (suite == "faco")
            compare_faco(settings, coordinates, instance, directory);
        else if (suite == "ils")
            compare_ils(settings, coordinates, instance, directory);
        else
            run_checks(settings, coordinates, instance, directory);
        cout << "Saved: " << fs::absolute(directory).string() << endl;
        return 0;
    } catch (const exception &error) {
        cerr << error.what() << '\n';
        return 1;
    }
}


## 7. Fetch dependencies and compile
Only the pinned FACO dependency is downloaded, for shared parsing and local search.
Its author/license notices remain intact. Setup and compilation use shell commands.
The project sources use `<bits/stdc++.h>` and `using namespace std;`.


In [ ]:
%%writefile /content/parallel_tsp_cpp/setup.sh
#!/usr/bin/env bash
set -euo pipefail
cd -- "$(dirname -- "$0")"

faco_revision=05b7c1b8d52b27d121b6138222f986fcbf196863
for command in g++ curl tar; do
    command -v "$command" >/dev/null || { echo "Missing: $command" >&2; exit 1; }
done
mkdir -p deps
if [[ -d deps/faco ]]; then
    [[ -f deps/faco/revision && $(cat deps/faco/revision) == "$faco_revision" ]] || {
        echo "Existing dependency has an unknown revision; use a fresh project directory." >&2
        exit 1
    }
else
    download_directory=$(mktemp -d)
    trap 'rm -rf -- "$download_directory"' EXIT
    curl --fail --location --retry 3 --max-time 120 \
        "https://codeload.github.com/RSkinderowicz/FocusedACO/tar.gz/$faco_revision" \
        -o "$download_directory/faco.tar.gz"
    mkdir "$download_directory/source"
    tar -xzf "$download_directory/faco.tar.gz" --strip-components=1 \
        -C "$download_directory/source"
    printf '%s\n' "$faco_revision" > "$download_directory/source/revision"
    mv -- "$download_directory/source" deps/faco
fi
echo "Pinned FACO dependencies ready. No Python packages are needed."


In [ ]:
%%writefile /content/parallel_tsp_cpp/build.sh
#!/usr/bin/env bash
set -euo pipefail
cd -- "$(dirname -- "$0")"

source_directory=deps/faco/src
compiler_flags=(-std=c++17 -O3 -march=native -fopenmp -D_USE_MATH_DEFINES
                -include cstdint -include cstring -I "$source_directory")
shared_sources=("$source_directory/problem_instance.cpp"
                "$source_directory/local_search.cpp"
                "$source_directory/utils.cpp"
                "$source_directory/rand.cpp")

g++ "${compiler_flags[@]}" -DNDEBUG faco.cpp "${shared_sources[@]}" -o faco
# Keep assertions enabled for the ILS perturbation/refinement self-test.
g++ "${compiler_flags[@]}" focused_sharing.cpp "${shared_sources[@]}" -o focused_sharing
g++ "${compiler_flags[@]}" experiments.cpp -o experiments
echo "Built the FACO solver, ILS solver and C++ experiment runner."


In [ ]:
%%bash
set -euo pipefail
cd /content/parallel_tsp_cpp
bash setup.sh
bash build.sh


## 8. Compare FACO baseline versus A+B
Both have identical iterations, ants, seeds and threads. Solver time excludes
candidate preprocessing; process time includes it. Speedup = baseline / A+B:
above 1 means A+B is faster. Add `--smoke` for a short integration check.


In [ ]:
%%bash
set -euo pipefail
cd /content/parallel_tsp_cpp
./experiments faco


## 9. Validate ILS and thread consistency
Checks permutations, duplicate coordinates, perturbations and refinement, rejects
malformed tours, and compares final states after 1,000 fixed rounds with one and
multiple threads. Both normal ILS and repair/sharing are checked. These correctness
runs are excluded from the following timed comparison.


In [ ]:
%%bash
set -euo pipefail
cd /content/parallel_tsp_cpp
./experiments check


## 10. Compare normal ILS versus ILS + focused repair + selective sharing
Each pair uses the same seed, threads and time cutoff. The table shows mean/best
tour cost, solver time and process time. Both include preprocessing in the solver
budget. An in-progress refinement may finish after the cutoff; its late result is
discarded, but external elapsed time includes this overrun.

Equal-time runs measure solution quality, **not speed to the same solution**.
Jobs completed are different-sized operations, so a jobs ratio is not a speedup.
Multiple seeds and larger budgets are needed for meaningful conclusions.


In [ ]:
%%bash
set -euo pipefail
cd /content/parallel_tsp_cpp
./experiments ils


## 11. Optional results download
Raw JSON, console logs, settings and CSV tables are inside `results/`.
Run this cell, then download `results.tar.gz` through Colab's Files sidebar.
Runtime files are temporary, so download results you want to keep.


In [ ]:
%%bash
set -euo pipefail
cd /content/parallel_tsp_cpp
tar -czf results.tar.gz results
